# `kfre`: Worked Examples

This notebook demonstrates the `kfre` library end to end on the public Ali et al. (2021) advanced-CKD cohort: single-patient and batch risk estimation with the Kidney Failure Risk Equation (KFRE), unit conversion, uPCR-to-uACR derivation, outcome labeling, performance evaluation, and CKD staging.

**KFRE in one line:** for a patient with chronic kidney disease, the equation estimates the probability of kidney failure (initiation of maintenance dialysis or kidney transplantation) within 2 and 5 years, using 4, 6, or 8 routine clinical variables.


In [ ]:
import pandas as pd
from kfre import kfre_person

In [ ]:
# Output paths for the performance figures saved later in this notebook
image_path_png = "../images/kfre_performance.png"
image_path_svg = "../images/kfre_performance.svg"

## 1. Single-Patient Risk

`kfre_person` computes the KFRE risk for one patient. Only the four base inputs are required (age, sex, eGFR, uACR); supplying diabetes/hypertension selects the 6-variable model, and supplying the four serum biochemical values selects the 8-variable model.

In [ ]:
# 4-variable, 2-year risk for a single patient
risk_percentage = kfre_person(
    age=57.28,
    is_male=False,
    eGFR=15.0,               # mL/min/1.73 m^2
    uACR=1762.001840,        # mg/g
    is_north_american=False,
    years=2,
    dm=None,
    htn=None,
    albumin=None,
    phosphorous=None,
    bicarbonate=None,
    calcium=None,
) * 100  # convert probability to a percentage

print(f"The 2-year risk of kidney failure for this patient is {risk_percentage:.2f}%.")

### 2- and 5-Year Risk

The same patient evaluated at both published KFRE horizons. Uncomment the optional blocks to switch from the 4-variable model to the 6- or 8-variable model.

In [ ]:
for years in [2, 5]:
    risk_percentage = (
        kfre_person(
            age=57.28,
            is_male=False,  # is the patient male?
            eGFR=15.0,  # ml/min/1.73 m^2
            uACR=1762.001840,  # mg/g
            is_north_american=False,  # is the patient from North America?
            years=years,
            ################################################################
            # Uncomment "dm" and "htn" for the 6-variable model:
            ################################################################
            # dm=0,
            # htn=1,
            ################################################################
            # Comment out "dm" and "htn"; uncomment the following lines for
            # the 8-variable model:
            ################################################################
            # albumin=3.0, # g/dL
            # phosphorous=3.162, # mg/dL
            # bicarbonate=21.3, # mEq/L
            # calcium=9.72, # mg/dL
        )
        * 100  # multiply by 100 to convert to percentage
    )

    message = f"The {years}-year risk of kidney failure for this patient is"
    print(f"{message} {risk_percentage:.2f}%.")

## 2. Converting Clinical Units

The Ali cohort reports laboratory values in SI units (mmol/L, g/L, mg/mmol), while the KFRE expects conventional units (mg/dL, g/dL, mg/g). `perform_conversions` handles this; with `reverse=False` it converts SI to conventional.

In [ ]:
from kfre import perform_conversions

In [ ]:
# Load the public Ali et al. (2021) advanced-CKD cohort
df = pd.read_csv("../data/12882_2021_2402_MOESM8_ESM.csv")

In [ ]:
# Convert SI -> conventional units for all recognized lab columns.
# New columns (e.g. Calcium_mg_dl, Albumin_g_dl, uPCR_mg_g) are added;
# the originals are preserved.
df = perform_conversions(
    df=df,
    reverse=False,
    convert_all=True,
)

df

## 3. Deriving uACR from uPCR

Where a measured urinary albumin-to-creatinine ratio (uACR) is unavailable, it can be estimated from the urinary protein-to-creatinine ratio (uPCR) using the Sumida et al. (2020) equation.

> **Note:** this conversion is an approximation. The albumin fraction of total
> urinary protein varies between individuals, so converted uACR values carry
> inherent measurement error and should be interpreted with caution. A directly
> measured uACR is preferred when available.

In [ ]:
from kfre import upcr_uacr

In [ ]:
# Estimate uACR (mg/g) from the converted uPCR column
df["uACR"] = upcr_uacr(
    df=df,
    sex_col="SEX",
    diabetes_col="Diabetes (1=yes; 0=no)",
    hypertension_col="Hypertension (1=yes; 0=no)",
    upcr_col="uPCR_mg_g",
    female_str="Female",
)

In [ ]:
df["uACR"]

## 4. Labeling Kidney-Failure Outcomes

`class_esrd_outcome` builds a binary outcome label for a chosen time horizon from a user-supplied kidney-failure event indicator (here the cohort's `ESRD` column) and a follow-up-duration column. It does **not** derive the event from eGFR.

### Right-censoring (sensitivity analysis)

Fixed-horizon labeling marks every patient without a recorded event as 0, including patients whose follow-up ended before the horizon (death without kidney failure, or loss to follow-up). Those patients are censored, not confirmed event-free. Setting `censor_incomplete=True` labels such patients `NaN` so they can be excluded, letting us check whether performance is robust to the censoring concern. Complete-case exclusion is a pragmatic guard rather than a full competing-risks model; the KFRE itself was derived with censoring-aware Cox models (Tangri et al.).


In [ ]:
from kfre import class_esrd_outcome

In [ ]:
# How many patients would be censored (incomplete follow-up, no event)?
for y in (2, 5):
    out = class_esrd_outcome(
        df.copy(), col="ESRD", years=y,
        duration_col="Follow-up YEARS", prefix="esrd",
        create_years_col=False, censor_incomplete=True,
    )
    n_nan = out[f"esrd_{y}_year_outcome"].isna().sum()
    print(f"{y}-year: {n_nan} patients censored (NaN)")

**Primary (naive) labeling**: reported in the manuscript. All non-events are labeled 0.

In [ ]:
# Primary labeling: event within the horizon -> 1, otherwise 0
for y in (2, 5):
    df = class_esrd_outcome(
        df=df, col="ESRD", years=y, duration_col="Follow-up YEARS",
        prefix="esrd", create_years_col=False,
    )

**Censored labeling**: sensitivity analysis on a separate copy. Non-event patients with follow-up shorter than the horizon are set to `NaN`.

In [ ]:
# Censored labeling: excludes (NaN) non-event patients with incomplete follow-up
df_censor_incomplete = df.copy()
for y in (2, 5):
    df_censor_incomplete = class_esrd_outcome(
        df=df_censor_incomplete, col="ESRD", years=y,
        duration_col="Follow-up YEARS", prefix="esrd",
        create_years_col=False, censor_incomplete=True,
    )

In [ ]:
# Naive labeling: 2- and 5-year outcome columns (all non-events = 0)
df.head()

In [ ]:
# Censored labeling: incomplete-follow-up non-events are NaN
df_censor_incomplete.head()

## 5. Batch Risk Across a Cohort

`add_kfre_risk_col` appends KFRE risk columns for every requested model variant and horizon to the DataFrame in one pass (columns named `kfre_{n}var_{y}year`).

In [ ]:
from kfre import add_kfre_risk_col

In [ ]:
# Compute KFRE risk for all variants (4/6/8) and horizons (2/5 yr).
# Each row gains columns kfre_4var_2year, kfre_4var_5year, ... kfre_8var_5year.
df = add_kfre_risk_col(
    df=df,
    age_col="Age",
    sex_col="SEX",
    eGFR_col="eGFR-EPI",
    uACR_col="uACR",
    dm_col="Diabetes (1=yes; 0=no)",
    htn_col="Hypertension (1=yes; 0=no)",
    albumin_col="Albumin_g_dl",
    phosphorous_col="Phosphate_mg_dl",
    bicarbonate_col="Bicarbonate (mmol/L)",
    calcium_col="Calcium_mg_dl",
    num_vars=[4, 6, 8],
    years=(2, 5),
    is_north_american=False,
    copy=False,
)

## 6. Performance Assessment

`eval_kfre_metrics` tabulates discrimination metrics, and `plot_kfre_metrics` draws the ROC and precision-recall curves for every model/horizon combination.

In [ ]:
from kfre import plot_kfre_metrics, eval_kfre_metrics

In [ ]:
# Inspect the outcome and prediction columns, and their missingness,
# before evaluating.
outcome_cols = [c for c in df.columns if 'year_outcome' in c]
kfre_cols = [c for c in df.columns if 'kfre_' in c]
print("outcome columns:", outcome_cols)
print("prediction columns:", kfre_cols)
print(df[outcome_cols + kfre_cols].isnull().sum())

In [ ]:
# Drop rows with missing predictions before scoring.
# (For the censored sensitivity analysis, evaluate df_censor_incomplete instead
#  and also drop rows with NaN outcome labels per horizon.)
kfre_cols = [c for c in df.columns if 'kfre_' in c]
df = df.dropna(subset=kfre_cols)

metrics_df_n_var = eval_kfre_metrics(
    df=df,
    n_var_list=[4, 6, 8],
    outcome_years=[2, 5],
)
metrics_df_n_var

In [ ]:
from kfre import bootstrap_metric_ci

metric_labels = {
    "auc_roc": "AUC ROC",
    "average_precision": "Average Precision",
    "precision": "Precision/PPV",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "brier": "Brier Score",
}

metrics = ["precision", "average_precision", "sensitivity",
           "specificity", "auc_roc", "brier"]

records = []
for n in [4, 6, 8]:
    for y in [2, 5]:
        y_true  = df[f"esrd_{y}_year_outcome"]
        y_score = df[f"kfre_{n}var_{y}year"]
        col = f"{y}_year_{n}_var_kfre"          # matches your existing column style
        for m in metrics:
            r = bootstrap_metric_ci(
                y_true, y_score, metric=m, n_boot=1000, seed=42, progress=True,
            )
            records.append({
                "Outcome Metrics": metric_labels[m],
                "column": col,
                "cell": f"{r['point']:.3f} ({r['lower']:.3f}\u2013{r['upper']:.3f})",
            })

ci_df = pd.DataFrame(records)

# Pivot to metrics-as-rows, model/horizon-as-columns
ci_table = (
    ci_df.pivot(index="Outcome Metrics", columns="column", values="cell")
    # keep the metric row order you use in eval_kfre_metrics
    .reindex([metric_labels[m] for m in metrics])
)

# order columns the same way as your eval_kfre_metrics table
col_order = [f"{y}_year_{n}_var_kfre" for n in [4, 6, 8] for y in [2, 5]]
ci_table = ci_table[[c for c in col_order if c in ci_table.columns]]

In [ ]:
ci_table

In [ ]:
from kfre import plot_kfre_metrics

In [ ]:
plot_kfre_metrics(
    df=df,                       # DataFrame to produce plots for
    num_vars=[4, 6, 8],          # 4,6,8 KFRE variables
    figsize=[6, 6],             # Custom figure size
    mode="plot",                 # Can be 'prep', 'plot', or 'both'
    image_prefix="performance",  # Optional prefix for saved images
    bbox_inches="tight",         # Bounding box in inches for the saved images
    plot_type="all_plots",       # Can be 'auc_roc', 'precision_recall', or 'all_plots'
    show_years=[2, 5],           # Year outcomes to show in the plots
    plot_combinations=True,      # Plot combinations of all variables in one plot
    show_subplots=True,          # Place all plots on one subplot; False does individual
    decimal_places=3,            # Number of decimal places in legend
    image_path_png = image_path_png,
    image_path_svg = image_path_svg,
    image_filename = "kfre_performance"
)

## 7. CKD Staging and Risk Figures

`class_ckd_stages` assigns KDIGO GFR-category labels from eGFR. The cells below regenerate the manuscript figures: predicted 2-year risk by CKD stage, and a comparison of the 4- vs 8-variable estimates.

In [ ]:
import matplotlib.pyplot as plt
from kfre import class_ckd_stages

# Assign CKD stages
df = class_ckd_stages(
    df=df,
    egfr_col="eGFR-EPI",
    stage_col="stage",
    combined_stage_col="stage_combined"
)

# Figure 1: Risk by CKD stage
fig, ax = plt.subplots(figsize=(8, 6))
stage_order = ["CKD Stage 4", "CKD Stage 5"]
data_by_stage = [
    df[df["stage"] == s]["kfre_4var_2year"].dropna().clip(lower=0)
    for s in stage_order
]
ax.boxplot(data_by_stage, tick_labels=["Stage 4", "Stage 5"])
ax.set_ylim(bottom=0)
ax.set_xlabel("CKD Stage")
ax.set_ylabel("Predicted 2-Year Risk")
ax.set_title("KFRE 4-Variable, 2-Year Risk by CKD Stage")
plt.tight_layout()
plt.savefig("../images/risk_by_stage.svg")
plt.show()

# Figure 2: 4-var vs 8-var scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df["kfre_4var_2year"], df["kfre_8var_2year"], alpha=0.5, s=10)
lims = [0, max(df["kfre_4var_2year"].max(), df["kfre_8var_2year"].max())]
ax.plot(lims, lims, "--", color="gray")
ax.set_xlabel("4-Variable KFRE (2-year)")
ax.set_ylabel("8-Variable KFRE (2-year)")
ax.set_title("4- vs. 8-Variable KFRE: 2-Year Risk Estimates")
plt.tight_layout()
plt.savefig("../images/4var_vs_8var.svg")
plt.show()

In [ ]:
print(df["stage"].value_counts())